# Part 3: Intelligence Layers

RAG with LCEL, CAG with a semantic cache and CRAG with corrective retrieval.

In [1]:
from prime_land.chat_service.cag_cache import CAGCache
from prime_land.chat_service.cag_service import CAGService
from prime_land.chat_service.crag_service import CRAGService
from prime_land.chat_service.rag_service import RAGService
from prime_land.config import CONFIG, get_path, load_faqs
from prime_land.ingest_document.indexer import get_client, open_store
from prime_land.providers import get_embeddings, get_llm

strategy = CONFIG["retrieval"]["strategy"]
embeddings = get_embeddings()
llm = get_llm()
client = get_client()
store = open_store(client, embeddings, strategy)
rag = RAGService(store, embeddings, llm, strategy=strategy)

## RAG

In [2]:
result = rag.generate("Which gated house community is available in Kalutara?")
print(result["answer"])
print(result["evidence_urls"])

- **Kaloora – Kalutara** – a master‑planned gated residential community with two‑storey homes, 24‑hour security, a swimming pool, gym, children’s play area, indoor cricket facility and multifunctional room.  
- Located just 5 minutes from Kalutara Town, 800 m from Nagoda Hospital and 15 minutes from the Dodangoda Highway entrance.  

**Summary:** Kaloora is the only gated house community currently available in Kalutara, offering modern amenities and secure living in a rapidly growing area.
['https://www.primelands.lk/house/KALOORA-KALUTARA/en', 'https://www.primelands.lk/news/en', 'https://www.primelands.lk/house/en', 'https://www.primelands.lk/en']


## CAG

In [3]:
cache = CAGCache(embeddings, get_path("cache_dir"), CONFIG["cag"]["similarity_threshold"], CONFIG["cag"]["history_ttl_hours"])
cag = CAGService(rag, cache)
print(cag.warm_faqs(load_faqs()), "FAQs warmed")

warming FAQ 1/12: What lands are available at Prime Lands?
warming FAQ 2/12: What houses are available at Prime Lands?
warming FAQ 3/12: What apartments are available at Prime Lands?
warming FAQ 4/12: How can I contact Prime Lands?
warming FAQ 5/12: What payment plans are available for lands?
warming FAQ 6/12: Do you offer bank loan facilities?
warming FAQ 7/12: What is the reservation fee for a land?
warming FAQ 8/12: Which gated house communities do you have?
warming FAQ 9/12: Are there lands in Kadawatha?
warming FAQ 10/12: Are there houses in Thalawathugoda?
warming FAQ 11/12: What upcoming projects does Prime Lands have?
warming FAQ 12/12: How do I sell my land to Prime Lands?
12 FAQs warmed


In [4]:
first = cag.generate("What lands are available at Prime Lands?")
second = cag.generate("Tell me what lands Prime Lands has")
for result in (first, second):
    print(result["cache_hit"], result["cache_source"], round(result["similarity"], 3), round(result["generation_time"], 3))
print(cag.stats())

True faq 1.0 0.0
True faq 0.915 0.475
{'total_queries': 2, 'hits': 2, 'misses': 0, 'faq_hits': 2, 'history_hits': 0, 'hit_rate': 1.0}


## CRAG

In [5]:
crag = CRAGService(rag, llm)
for question in ["Which gated house community is available in Kalutara?", "I want a quiet land by the beach in the south"]:
    result = crag.generate(question)
    print(question)
    print(round(result["confidence_initial"], 3), "->", round(result["confidence_final"], 3), "corrected:", result["correction_applied"])

Which gated house community is available in Kalutara?
0.798 -> 0.798 corrected: False
I want a quiet land by the beach in the south
0.564 -> 0.582 corrected: True


In [6]:
from prime_land.chat_service.utils import calculate_confidence, confidence_parts

questions = [item["question"] for item in CONFIG["evaluation"]["queries"] + CONFIG["evaluation"]["hard_queries"]]
for question in questions:
    docs = rag.retrieve(question)
    keyword, similarity = confidence_parts(docs, question)
    mean_score = sum(doc.metadata["score"] for doc in docs) / len(docs)
    confidence = calculate_confidence(docs, question)
    print(f"conf {confidence:.2f} keyword {keyword:.2f} similarity {similarity:.2f} mean_score {mean_score:.3f} | {question}")


conf 0.80 keyword 0.80 similarity 0.80 mean_score 0.709 | Which gated house community is available in Kalutara?
conf 0.76 keyword 0.75 similarity 0.77 mean_score 0.704 | What is the price of land at Solara Meepe?
conf 0.85 keyword 1.00 similarity 0.70 mean_score 0.690 | Tell me about Veloria in Bokundara
conf 0.67 keyword 0.33 similarity 1.00 mean_score 0.760 | Which houses are available in Thalawathugoda?
conf 0.64 keyword 0.67 similarity 0.61 mean_score 0.672 | What amenities does Kaloora offer?
conf 0.79 keyword 0.67 similarity 0.92 mean_score 0.734 | Are there lands near Galle?
conf 0.74 keyword 0.67 similarity 0.82 mean_score 0.714 | What is the payment plan for Kaloora?
conf 0.51 keyword 0.75 similarity 0.26 mean_score 0.602 | Where is Scottish Island located?
conf 0.92 keyword 1.00 similarity 0.84 mean_score 0.717 | Which land is available in Kadawatha?
conf 0.96 keyword 1.00 similarity 0.91 mean_score 0.732 | How can I contact Prime Lands?
conf 0.70 keyword 0.83 similarity 0.56

In [7]:
client.close()